# ✅ AIC — Judge Quickstart (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/COolAlien35/AIC/blob/main/judge_colab.ipynb)

This notebook is the **fastest way to verify the submission works**.

It has two parts:

1. **No-install remote check** (recommended): hit the public Hugging Face Space and verify the required OpenEnv endpoints.
2. **Optional local run**: clone the repo, install dependencies, and run a tiny benchmark locally.

> Canonical judge Space:
> - Space page: `https://huggingface.co/spaces/KINGKK007/aic-training`
> - Runtime: `https://kingkk007-aic-training.hf.space`


## 1) Remote verification (no installs)

This proves the Space is alive and exposes the expected API:

- `GET /health`
- `POST /reset`
- `GET /state/{env_id}`


In [ ]:
import json
import textwrap
from urllib.request import Request, urlopen

HOST = "https://kingkk007-aic-training.hf.space"

def get(path: str) -> dict:
    with urlopen(HOST + path, timeout=10) as r:
        return json.loads(r.read().decode())

def post(path: str, payload: dict) -> dict:
    data = json.dumps(payload).encode()
    req = Request(HOST + path, data=data, headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())

health = get("/health")
print("/health →", health)

reset = post("/reset", {"episode_id": 0, "base_seed": 42, "fault_mode": "cascading_failure"})
env_id = reset.get("env_id")
print("/reset env_id →", env_id)

assert env_id, "reset did not return env_id"

state = get(f"/state/{env_id}")
print("/state keys →", list(state.keys()))
print("state.step →", state["state"].get("step"))
print("state.health_score →", state["state"].get("health_score"))

# A small, readable snippet of the observation
obs = reset.get("observation", {})
print("\nObservation (snippet):")
print(textwrap.shorten(obs.get("alert_summary_text", ""), width=220, placeholder=" …"))

## 2) Optional: local clone + tiny run

This is optional, but it proves the repo can be cloned and executed locally.

> Note: this installs the full Python dependencies, which can take a few minutes on Colab.


In [ ]:
%%bash
set -euo pipefail

if [ ! -d "/content/AIC" ]; then
  git clone https://github.com/COolAlien35/AIC.git /content/AIC
fi
cd /content/AIC

python -m pip install -q --upgrade pip
python -m pip install -q -r requirements.txt

# Fast CPU-safe checks
python scripts/run_pipeline.py verify
python scripts/run_final_benchmark.py --episodes 1
python scripts/score_tasks.py --episodes 1

echo "\nDone."